In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
df = pd.read_csv("gk_qna_dataset.csv")

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df['question'].unique()

# tokenization with Regular experssion

In [ ]:

def tokenize(text):

  # convert all the test in to the lower case
  text = text.lower()
  #remove all the quotes
  text = re.sub(r"[\"']","",text)
  #remove all the punctution
  text = re.sub(r"[^a-z0-9\s]"," ", text)
  #remove the extra spaces
  text = re.sub(r"[/s+]"," ",text).strip()
  #split all the tokens
  tokens = text.split()

  return tokens



In [ ]:
print(tokenize(df['question'][0]))

Vocab forming

In [ ]:
vocab = {'<UNK>' : 0}

In [ ]:
def build_vocab(row):
  #print(row['question'],row['answer'])

  tokenize_question = tokenize(row['question'])
  tokenize_answer = tokenize(row['answer'])

  #print(tokenize_question,tokenize_answer)

  marged_tokens = tokenize_question + tokenize_answer

  print(marged_tokens)

  for token in marged_tokens:
    if token is not vocab:
      vocab[token] = len(token)

In [ ]:
df.apply(build_vocab,axis=1)

In [ ]:
vocab

In [ ]:
len(vocab)

In [ ]:
def text_to_indexs(text,vocab):
  indexed_text = []

  for token in tokenize(text):
    #print(token)

    if token in vocab:
      #print(vocab[token])
      indexed_text.append(vocab[token])

    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [ ]:
text_to_indexs("what is school",vocab)

In [ ]:
df.shape[0]

# Dataset and DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

In [ ]:
index = 0

text_to_indexs(df.iloc[index]['question'],vocab)

In [ ]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numercial_question = text_to_indexs(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indexs(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numercial_question) , torch.tensor(numerical_answer)

In [ ]:
dataset = QADataset(df,vocab)


In [ ]:
dataset[3]

In [ ]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)